## MFL Deep Learning Engine: Initialization & Setup
Loading required libraries, setting up global variables, and defining paths for datasets and model weights.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import random
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('../')
from src.ml.dataset import get_single_dataloaders, get_multi_dataloaders
from src.ml.train_model import MFLTrainer
from src.ml.evaluator import MFLEvaluator, ComprehensiveFusionEvaluator
from src.ml.fusion_engine import MFLFusionEngine


sns.set_theme(style="darkgrid", palette="deep")

SINGLE_DATA = "../data/processed/single_dataset.npz"
MULTI_DATA = "../data/processed/multi_dataset.npz"
SINGLE_MODEL = "../models/single_mfl_net.pth"
MULTI_MODEL = "../models/multi_mfl_net.pth"

# Load base test sets and select fixed random samples for consistent plotting across cells
_, single_test = get_single_dataloaders(SINGLE_DATA, batch_size=1)
_, multi_test = get_multi_dataloaders(MULTI_DATA, batch_size=1)

single_samples = random.sample(list(single_test), 5)
multi_samples = random.sample(list(multi_test), 5)


## 1. Train Base Models (Single & Multi)
Executes the training loop for both 1D-CNN architectures. (Skip this cell if models are already trained and weights are saved).

In [ ]:
trainer = MFLTrainer(
    single_data_path=SINGLE_DATA,
    multi_data_path=MULTI_DATA,
    batch_size=128,
    epochs=100,
    lr=0.001
)

print("Starting training sequence...")
# trainer.train_single_model()
# trainer.train_multi_model()
# trainer.save_models(single_save_path=SINGLE_MODEL, multi_save_path=MULTI_MODEL)
print("Models trained and weights saved successfully.")

## 2. Evaluate Base Models
Generates probabilistic evaluation metrics (Calibration, ROC, Confidence Histograms) and Engineering Gap Tolerance matrices for the isolated models.

In [ ]:
print("Evaluating Isolated Base Models...")
evaluator = MFLEvaluator(SINGLE_DATA, MULTI_DATA, SINGLE_MODEL, MULTI_MODEL)
evaluator.evaluate_models(temperature=3.0)

# Plotting metrics 
evaluator.plot_probabilistic_metrics()
evaluator.plot_gap_tolerance()

# Generate the 10 overlay plots (5 Single, 5 Multi) for Base Models
print("\nGenerating Top-3 Probabilistic Overlays for Base Models...")
evaluator.plot_base_top3(single_samples, multi_samples, temperature=3.0)

## 3. Fusion Engine Analytics
Runs the Uncertainty-Aware Fusion Engine on a test subset, applying class-conditional weighted averages. Outputs combined probabilistic metrics and Top-3 physical signal reconstructions.

In [ ]:
print("Initializing Massive Fusion Analytics (500 samples)...")
fusion_eval = ComprehensiveFusionEvaluator(SINGLE_MODEL, MULTI_MODEL)
fusion_eval.run_statistical_evaluation(single_test, multi_test, num_samples=500)

# Plots Fusion Metrics (ROC, Calibration, Tolerance Matrix)
fusion_eval.plot_comprehensive_metrics()

#Generate the 10 overlay plots (5 Single, 5 Multi) for the Fusion Engine
print("\nGenerating Top-3 Probabilistic Overlays for Fusion Engine...")
fusion_eval.plot_fusion_top3(single_samples, multi_samples)

# Run and plot the Top-K (Pie chart, Bar chart, etc.) analytics
print("\nRunning Probabilistic Analytics on 100 Samples (Fusion Engine)...")
fusion_eval.plot_top_k_bar_chart(single_test, multi_test, num_samples=400)
fusion_eval.plot_macro_confusion_matrix()

In [ ]:
fusion_eval.plot_fusion_top3(single_samples, multi_samples)